# Hyper-Sonic Performance Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob
import os
import shutil
from google.colab import drive

# Set a consistent visual style and color palette
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({
    "font.family": "serif",
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 16
})

In [ ]:
# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("Mounting Google Drive...")
    try:
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted successfully.")
    except Exception as e:
        print(f"Error mounting Google Drive: {e}")
else:
    print("Google Drive is already mounted.")

Google Drive is already mounted.


In [ ]:
TARGET_FILE = 'aero_sweep_results.csv' # Original file name

In [ ]:
def split_massive_csv(input_file, target_size_mb=24):
    """
    Splits the CSV into chunks smaller than the target size (24MB for safety).
    Estimates row count based on file size to stay under the limit.
    """
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return

    # Estimate rows per chunk to stay under ~24MB
    file_size = os.path.getsize(input_file)
    # Sample first few lines to get average row size
    temp_df = pd.read_csv(input_file, nrows=1000)
    avg_row_size = temp_df.memory_usage(deep=True).sum() / 1000
    rows_per_chunk = int((target_size_mb * 1024 * 1024) / avg_row_size)

    output_dir = "split_chunks"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Splitting {input_file} into ~{target_size_mb}MB chunks...")
    reader = pd.read_csv(input_file, chunksize=rows_per_chunk)

    for i, chunk in enumerate(reader):
        chunk_filename = f"{output_dir}/aero_part_{i+1:03d}.csv"
        chunk.to_csv(chunk_filename, index=False)
        mb = os.path.getsize(chunk_filename) / (1024 * 1024)
        print(f"Generated {chunk_filename}: {mb:.2f} MB")

In [ ]:
# Call the function to split the CSV file
# Using the default rows_per_chunk=1000000, which typically results in files 40-60MB,
# satisfying the user's request for files 100MB or lower.
split_massive_csv(TARGET_FILE)

Splitting aero_sweep_results.csv into ~24MB chunks...
Generated split_chunks/aero_part_001.csv: 21.38 MB
Generated split_chunks/aero_part_002.csv: 21.38 MB
Generated split_chunks/aero_part_003.csv: 21.38 MB
Generated split_chunks/aero_part_004.csv: 21.38 MB
Generated split_chunks/aero_part_005.csv: 21.38 MB
Generated split_chunks/aero_part_006.csv: 21.38 MB
Generated split_chunks/aero_part_007.csv: 21.38 MB
Generated split_chunks/aero_part_008.csv: 21.38 MB
Generated split_chunks/aero_part_009.csv: 21.38 MB
Generated split_chunks/aero_part_010.csv: 21.38 MB
Generated split_chunks/aero_part_011.csv: 21.38 MB
Generated split_chunks/aero_part_012.csv: 21.38 MB
Generated split_chunks/aero_part_013.csv: 21.38 MB
Generated split_chunks/aero_part_014.csv: 21.38 MB
Generated split_chunks/aero_part_015.csv: 21.38 MB
Generated split_chunks/aero_part_016.csv: 20.25 MB


In [ ]:
def load_aero_data():
    """
    Loads all aero data chunks from the 'split_chunks' directory into a single DataFrame.
    """
    print("Searching for data shards...")
    file_pattern = "split_chunks/aero_data_part_*.csv"
    files = glob.glob(file_pattern)

    if not files:
        print("No files found matching 'aero_data_part_*.csv'. Please ensure the 'split_chunks' directory exists and contains the CSV parts.")
        return None

    print(f"Loading {len(files)} files into memory...")
    # Sort files to ensure consistent order of concatenation
    df = pd.concat([pd.read_csv(f) for f in sorted(files)], ignore_index=True)
    print(f"Dataset Loaded: {len(df):,} total trials.")
    return df

In [ ]:
def ensure_output_dir(base_dir="analysis_plots", sub_dir=""):
    """Ensures the output directory exists."""
    path = os.path.join(base_dir, sub_dir)
    os.makedirs(path, exist_ok=True)
    return path

In [ ]:
def generate_dashboard(df_data=None, g_filter=None, base_output_dir="analysis_plots"):
    """
    Generates a performance dashboard. Can operate on a full dataset or a filtered subset.

    Args:
        df_data (pd.DataFrame, optional): The DataFrame to use. If None, data will be loaded.
        g_filter (float, optional): If provided, filters the DataFrame to this specific 'g' value.
        base_output_dir (str): Base directory to save dashboards.
    """
    if df_data is None:
        df = load_aero_data()
        if df is None: return
    else:
        df = df_data.copy()

    dashboard_output_dir = ensure_output_dir(base_output_dir, "dashboards")

    if g_filter is not None:
        df_filtered = df[df['g'] == g_filter].copy()
        if df_filtered.empty:
            print(f"No data found for G = {g_filter}. Skipping dashboard generation.")
            return
        dashboard_title = f'AV-32 Hyper-Sonic Performance Analysis (G={g_filter})\nTotal Trials: {len(df_filtered):,}'
        output_filename = f"Aero_Performance_Dashboard_G{g_filter}.png"
        data_to_plot = df_filtered # Use filtered data for plots
    else:
        dashboard_title = f'AV-32 Hyper-Sonic Performance Analysis\nTotal Trials: {len(df):,}'
        output_filename = "Aero_Performance_Dashboard.png"
        data_to_plot = df # Use full data for plots

    # Create figure with 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(dashboard_title, fontsize=20, fontweight='bold')

    # GRAPH 1: EFFICIENCY DISTRIBUTION
    print("Generating distribution plot...")
    sns.histplot(data_to_plot['efficiency'], kde=True, color='teal', bins=50, ax=axes[0, 0])
    axes[0, 0].set_title('Efficiency Distribution', fontsize=14)
    axes[0, 0].set_xlabel(r'Efficiency (eta)')
    axes[0, 0].set_ylabel('Frequency')

    # GRAPH 2: GUST INTENSITY (G) VS EFFICIENCY (or Efficiency Distribution for single G) (Boxplot or Distribution)
    print("Generating second plot...")
    if g_filter is None:
        # Boxplot for overall dashboard
        sns.boxplot(x='g', y='efficiency', data=data_to_plot, hue='g', palette='viridis', legend=False, ax=axes[0, 1])
        axes[0, 1].set_title('Impact of Gust Intensity ($G$) on Efficiency', fontsize=14)
        axes[0, 1].set_xlabel('Gust Parameter ($G$)')
        axes[0, 1].set_ylabel(r'Efficiency (eta)')
    else:
        # Distribution for single G value
        sns.histplot(data_to_plot['efficiency'], kde=True, color='purple', bins=50, ax=axes[0, 1])
        axes[0, 1].set_title(f'Efficiency Distribution for G={g_filter}', fontsize=14)
        axes[0, 1].set_xlabel(r'Efficiency (eta)')
        axes[0, 1].set_ylabel('Frequency')

    # GRAPH 3: PERFORMANCE STABILITY (Moving Average)
    print("Generating stability plot...")
    # We take a sample or decimate for the line plot if it's 10M to save memory/rendering time
    step = max(1, len(data_to_plot) // 1000)
    sampled_df = data_to_plot.iloc[::step].copy()
    sampled_df['rolling_avg'] = sampled_df['efficiency'].rolling(window=50).mean()

    axes[1, 0].plot(sampled_df['rolling_avg'], color='orange', linewidth=2)
    axes[1, 0].set_title('System Stability (50-Window Rolling Mean)', fontsize=14)
    axes[1, 0].set_xlabel('Sampled Trial Chronology')
    axes[1, 0].set_ylabel(r'eta (Avg)')

    # GRAPH 4: CORRELATION MATRIX
    print("Generating correlation heatmap...")
    corr = data_to_plot.corr(numeric_only=True)
    sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, ax=axes[1, 1])
    axes[1, 1].set_title('Parameter Correlation Matrix', fontsize=14)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Save the result
    output_path = os.path.join(dashboard_output_dir, output_filename)
    plt.savefig(output_path, dpi=300)
    print(f"Dashboard saved as {output_path}")
    plt.close(fig) # Close plot to free memory

In [ ]:
def run_full_analysis():
    print("Starting detailed statistical analysis...")
    df = load_aero_data()
    if df is None: return []

    generated_plot_files = []
    output_dir = ensure_output_dir("analysis_plots", "full_analysis")

    # Optimization: Sample large data for trend plots to prevent memory crashes
    sample_size = 100000
    df_sampled = df.sample(n=min(len(df), sample_size)).sort_index() if len(df) > sample_size else df

    # 1. CONVERGENCE PLOT (Optimized with sampling)
    print("Plotting Efficiency Convergence...")
    plt.figure(figsize=(10, 6))
    df_sampled['rolling_mean'] = df_sampled['efficiency'].expanding().mean()
    plt.plot(df_sampled['rolling_mean'].values, color='#2c3e50', label='Expansion Mean (Sampled)')
    plt.axhline(y=df['efficiency'].mean(), color='r', linestyle='--', label='Final Global Mean')
    plt.title("Statistical Convergence of Efficiency")
    plt.xlabel("Trial Count (Sampled Index)")
    plt.ylabel("Cumulative Average Efficiency")
    plt.legend()
    f1 = os.path.join(output_dir, "convergence.png")
    plt.savefig(f1, dpi=300)
    plt.close()
    generated_plot_files.append(f1)

    # 2. EFFICIENCY BY GUST (Violin)
    print("Plotting Gust Intensity impact...")
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='g', y='efficiency', data=df_sampled, inner="quart", palette="viridis")
    plt.title("Efficiency Distribution by Gust Intensity (G)")
    f2 = os.path.join(output_dir, "efficiency_vs_gust.png")
    plt.savefig(f2, dpi=300)
    plt.close()
    generated_plot_files.append(f2)

    # 3. PARAMETER CORRELATION
    print("Plotting Correlations...")
    plt.figure(figsize=(8, 6))
    corr = df[['g', 'm', 'r', 'efficiency']].corr()
    sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0)
    plt.title("System Parameter Correlations")
    f3 = os.path.join(output_dir, "correlation_matrix.png")
    plt.savefig(f3, dpi=300)
    plt.close()
    generated_plot_files.append(f3)

    return generated_plot_files

In [ ]:
def save_to_drive(file_path, drive_folder_name='AeroAnalysis_Plots'):
    """
    Saves a file to a specified folder in Google Drive.
    The Google Drive must already be mounted.
    """
    drive_path = os.path.join('/content/drive/MyDrive', drive_folder_name)
    os.makedirs(drive_path, exist_ok=True)

    destination_path = os.path.join(drive_path, os.path.basename(file_path))
    shutil.copy(file_path, destination_path)
    print(f"Saved '{os.path.basename(file_path)}' to Google Drive: {destination_path}")

### Generate Dashboards and Run Full Analysis

In [ ]:
# FINAL AUTOMATED PIPELINE RE-RUN
import os
import glob
import pandas as pd

print("--- STEP 1: SPLITTING DATA ---")
split_massive_csv(TARGET_FILE, target_size_mb=24)

print("\n--- STEP 2: LOADING DATA ---")
def load_shards():
    file_pattern = "split_chunks/aero_part_*.csv"
    files = sorted(glob.glob(file_pattern))
    if not files:
        files = sorted(glob.glob("split_chunks/aero_data_part_*.csv"))
    if not files:
        print("No shards found.")
        return None
    print(f"Loading {len(files)} shards...")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

full_df = load_shards()

if full_df is not None:
    print(f"Dataset Loaded: {len(full_df):,} total trials.")
    print("\n--- STEP 3: GENERATING ANALYTICS ---")
    generated_files = []

    # Main Dashboard
    print("Generating Main Dashboard...")
    main_path = generate_dashboard(full_df)
    if main_path: generated_files.append(main_path)

    # Per-G Dashboards
    for g in sorted(full_df['g'].unique()):
        print(f"Processing G={g} Dashboard...")
        p = generate_dashboard(full_df, g_filter=g)
        if p: generated_files.append(p)

    print("\n--- STEP 4: UPLOADING TO DRIVE ---")
    folder_name = 'AeroAnalysis_Results'
    for f in generated_files:
        if f and os.path.exists(f):
            save_to_drive(f, drive_folder_name=folder_name)

    print(f"\nSUCCESS: Pipeline complete. Results uploaded to Google Drive folder '{folder_name}'")
else:
    print("Pipeline failed at loading stage.")

--- STEP 1: SPLITTING DATA ---
Splitting aero_sweep_results.csv into ~24MB chunks...
Generated split_chunks/aero_part_001.csv: 21.38 MB
Generated split_chunks/aero_part_002.csv: 21.38 MB
Generated split_chunks/aero_part_003.csv: 21.38 MB
Generated split_chunks/aero_part_004.csv: 21.38 MB
Generated split_chunks/aero_part_005.csv: 21.38 MB
Generated split_chunks/aero_part_006.csv: 21.38 MB
Generated split_chunks/aero_part_007.csv: 21.38 MB
Generated split_chunks/aero_part_008.csv: 21.38 MB
Generated split_chunks/aero_part_009.csv: 21.38 MB
Generated split_chunks/aero_part_010.csv: 21.38 MB
Generated split_chunks/aero_part_011.csv: 21.38 MB
Generated split_chunks/aero_part_012.csv: 21.38 MB
Generated split_chunks/aero_part_013.csv: 21.38 MB
Generated split_chunks/aero_part_014.csv: 21.38 MB
Generated split_chunks/aero_part_015.csv: 21.38 MB
Generated split_chunks/aero_part_016.csv: 20.25 MB

--- STEP 2: LOADING DATA ---
Loading 16 shards...
Dataset Loaded: 10,000,000 total trials.

--- ST

### Cleanup Instructions

Now that the data has been split, analyzed, and all generated plots have been moved to your Google Drive (in the `AeroAnalysis_Plots` folder), you can manually delete the previous cells in this notebook to clean it up. Look for the 'trash can' icon next to each cell when you hover over it. This includes any cells related to the previous `Vel` folder move or redundant function definitions.

# Move 'Vel' Folder to Google Drive

In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
# The previous cell failed because the mountpoint already contains files, likely meaning Drive is already mounted.
if not os.path.exists('/content/drive/MyDrive'):
    print("Mounting Google Drive...")
    try:
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted successfully.")
    except Exception as e:
        print(f"Error mounting Google Drive: {e}")
else:
    print("Google Drive is already mounted.")

Google Drive is already mounted.


In [ ]:
import shutil

source_folder = 'Vel'
destination_folder_name = 'Vel_Backup_from_Colab'

# Define the full path in Google Drive
drive_path = os.path.join('/content/drive/MyDrive', destination_folder_name)

if os.path.exists(source_folder):
    print(f"Moving '{source_folder}' to Google Drive at '{drive_path}'...")
    try:
        shutil.move(source_folder, drive_path)
        print(f"Successfully moved '{source_folder}' to '{drive_path}'.")
    except Exception as e:
        print(f"Error moving folder: {e}")
        print("Please check if the folder already exists in Google Drive or if there are permission issues.")
else:
    print(f"Source folder '{source_folder}' not found. Skipping move to Drive.")


Moving 'Vel' to Google Drive at '/content/drive/MyDrive/Vel_Backup_from_Colab'...
Successfully moved 'Vel' to '/content/drive/MyDrive/Vel_Backup_from_Colab'.


### Cleanup Instructions

Now that the `Vel` folder has been moved to your Google Drive, you can manually delete the previous cells in this notebook to clean it up. Look for the 'trash can' icon next to each cell when you hover over it.

In [ ]:
TARGET_FILE = 'aero_sweep_results.csv' # Assuming this is the original file name based on previous outputs

In [ ]:
import pandas as pd
import os

def split_massive_csv(input_file, rows_per_chunk=10000):
    """
    Splits a large CSV into multiple smaller files.
    Default 1,000,000 rows usually results in ~40MB to 60MB files
    depending on the number of columns.
    """
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return

    # Create an output directory
    output_dir = "split_chunks"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Starting split of {input_file}...")

    # Use pandas chunksize to process file without loading it all into RAM
    reader = pd.read_csv(input_file, chunksize=rows_per_chunk)

    for i, chunk in enumerate(reader):
        chunk_filename = f"{output_dir}/aero_data_part_{i+1}.csv"

        # Save chunk
        chunk.to_csv(chunk_filename, index=False)

        # Get file size for confirmation
        file_size_mb = os.path.getsize(chunk_filename) / (1024 * 1024)
        print(f"Saved: {chunk_filename} | Size: {file_size_mb:.2f} MB | Rows: {len(chunk)}")

    print(f"\nSuccess! All chunks are stored in the '{output_dir}' folder.")

print("split_massive_csv function defined and ready.")

split_massive_csv function defined and ready.


In [ ]:
# Re-run the split function to ensure data chunks are present
split_massive_csv(TARGET_FILE)

Starting split of aero_sweep_results.csv...
Saved: split_chunks/aero_data_part_1.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_2.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_3.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_4.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_5.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_6.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_7.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_8.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_9.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_10.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_11.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_12.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data_part_13.csv | Size: 0.34 MB | Rows: 10000
Saved: split_chunks/aero_data

In [ ]:
def load_aero_data():
    """
    Loads all aero data chunks from the 'split_chunks' directory into a single DataFrame.
    """
    print("Searching for data shards...")
    file_pattern = "split_chunks/aero_data_part_*.csv"
    files = glob.glob(file_pattern)

    if not files:
        print("No files found matching 'aero_data_part_*.csv'. Please check file names and directory.")
        return None

    print(f"Loading {len(files)} files into memory...")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f"Dataset Loaded: {len(df):,} total trials.")
    return df

In [ ]:
def generate_dashboard(df_data, g_filter=None, base_output_dir="analysis_plots"):
    """
    Generates robust dashboards with fixed LaTeX labels using double-escaped raw strings.
    """
    df = df_data if g_filter is None else df_data[df_data['g'] == g_filter].copy()
    if df.empty: return

    dashboard_output_dir = ensure_output_dir(base_output_dir, "dashboards")
    title = "Overall Performance" if g_filter is None else f"Performance (G={g_filter})"
    fname = "Dashboard_Overall.png" if g_filter is None else f"Dashboard_G{g_filter}.png"

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(title, fontsize=20, fontweight='bold')

    # Distribution - FIXED LaTeX using raw strings and double escape
    sns.histplot(df['efficiency'], kde=True, color='teal', ax=axes[0, 0])
    axes[0, 0].set_title('Efficiency Distribution')
    axes[0, 0].set_xlabel(r'Efficiency (eta)')

    # Boxplot / Per G - FIXED LaTeX
    if g_filter is None:
        sns.boxplot(x='g', y='efficiency', data=df, hue='g', palette='viridis', legend=False, ax=axes[0, 1])
        axes[0, 1].set_xlabel(r'Gust Parameter ($G$)')
    else:
        sns.kdeplot(df['efficiency'], fill=True, ax=axes[0, 1])
    axes[0, 1].set_ylabel(r'Metric (eta)')

    # Stability (Sampled)
    step = max(1, len(df) // 1000)
    axes[1, 0].plot(df['efficiency'].iloc[::step].rolling(50).mean().values, color='orange')
    axes[1, 0].set_title('System Stability')
    axes[1, 0].set_ylabel(r'Avg eta')

    # Correlation
    sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='RdBu_r', ax=axes[1, 1])

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    save_path = os.path.join(dashboard_output_dir, fname)
    plt.savefig(save_path, dpi=200)
    plt.close(fig)
    return save_path

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob

# Style settings for academic-quality plots
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 16
})

def run_full_analysis():
    # 1. LOAD AND MERGE DATA
    print("Loading data shards...")
    # Corrected file pattern to look into 'split_chunks' directory
    files = sorted(glob.glob("split_chunks/aero_data_part_*.csv"))
    if not files:
        print("Error: No CSV files found. Please ensure the 'split_chunks' directory exists and contains the CSV parts.")
        return

    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f"Dataset ready: {len(df):,} trials across {df['g'].nunique()} gust levels.")

    # 2. CONVERGENCE PLOT (Mean Efficiency over Time)
    print("Generating Figure 1: Convergence...")
    plt.figure(figsize=(10, 6))
    df['rolling_mean'] = df['efficiency'].expanding().mean()
    plt.plot(df['rolling_mean'], color='#2c3e50', label='Global Expansion Mean')
    plt.axhline(y=df['efficiency'].mean(), color='r', linestyle='--', label='Final Mean')
    plt.title(r"Statistical Convergence of Efficiency (eta)") # Fixed escape sequence
    plt.xlabel("Trial Count")
    plt.ylabel(r"Cumulative Average eta") # Fixed escape sequence
    plt.legend()
    plt.savefig("figG1_convergence.png", dpi=300)
    plt.close()

    # 3. ETA VS GUST INTENSITY (Violin + Boxplot)
    print("Generating Figure 2: Efficiency vs Gust...")
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='g', y='efficiency', data=df, inner="quart", palette="muted")
    plt.title(r"Distribution of Efficiency (eta) by Gust Intensity ($G$)") # Fixed escape sequence
    plt.xlabel(r"Gust Intensity ($G$)") # Fixed escape sequence
    plt.ylabel(r"Efficiency (eta)") # Fixed escape sequence
    plt.savefig("figG2_eta_vs_G.png", dpi=300)
    plt.close()

    # 4. KERNEL DENSITY ESTIMATE (KDE)
    print("Generating Figure 4: Density Analysis...")
    plt.figure(figsize=(10, 6))
    for g_val in sorted(df['g'].unique()):
        subset = df[df['g'] == g_val]
        sns.kdeplot(subset['efficiency'], label=f'G={g_val}', fill=True, alpha=0.1)
    plt.title("Efficiency Density Profiles across Gust Conditions")
    plt.xlabel(r"Efficiency (eta)") # Fixed escape sequence
    plt.ylabel("Probability Density")
    plt.legend(title="Gust Levels")
    plt.savefig("figG4_kde.png", dpi=300)
    plt.close()

    # 5. CORRELATION HEATMAP
    print("Generating Figure 6: Parameter Correlations...")
    plt.figure(figsize=(8, 6))
    corr = df[['g', 'm', 'r', 'efficiency']].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Parameter Correlation Matrix")
    plt.savefig("figG6_heatmap.png", dpi=300)
    plt.close()

    # 6. STABILITY ANALYSIS (Variance window)
    print("Generating Figure 7: Stability...")
    plt.figure(figsize=(10, 6))
    stability = df['efficiency'].rolling(window=1000).std()
    plt.plot(stability, color='darkgreen', alpha=0.7)
    plt.title("System Variance Stability (1000-Trial Rolling Std Dev)")
    plt.xlabel("Trial Index")
    plt.ylabel(r"Standard Deviation ($\sigma$)") # Fixed escape sequence
    plt.savefig("figG7_stability.png", dpi=300)
    plt.close()

    # 7. CUMULATIVE DISTRIBUTION FUNCTION (CDF)
    print("Generating Figure 8: CDF...")
    plt.figure(figsize=(10, 6))
    for g_val in sorted(df['g'].unique()):
        subset = df[df['g'] == g_val]
        sorted_data = np.sort(subset['efficiency'])
        yvals = np.arange(len(sorted_data)) / float(len(sorted_data)-1)
        plt.plot(sorted_data, yvals, label=f'G={g_val}')
    plt.title("Cumulative Distribution of Efficiency")
    plt.xlabel(r"Efficiency (eta)") # Fixed escape sequence
    plt.ylabel(r"Probability $P(X \leq x)$") # Fixed escape sequence
    plt.legend()
    plt.savefig("figG8_cdf.png", dpi=300)
    plt.close()

    print("\nAll analysis files generated successfully!")
    print("Files saved: figG1, figG2, figG4, figG6, figG7, figG8.")

if __name__ == "__main__":
    run_full_analysis()

Loading data shards...
Dataset ready: 10,000,000 trials across 5 gust levels.
Generating Figure 1: Convergence...
Generating Figure 2: Efficiency vs Gust...


/tmp/ipykernel_103096/2274381349.py:47: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='g', y='efficiency', data=df, inner="quart", palette="muted")


Generating Figure 4: Density Analysis...
Generating Figure 6: Parameter Correlations...
Generating Figure 7: Stability...
Generating Figure 8: CDF...


/tmp/ipykernel_103096/2274381349.py:100: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig("figG8_cdf.png", dpi=300)



All analysis files generated successfully!
Files saved: figG1, figG2, figG4, figG6, figG7, figG8.


In [ ]:
# Now, run the full analysis again
run_full_analysis()

Loading data shards...
Dataset ready: 10,000,000 trials across 5 gust levels.
Generating Figure 1: Convergence...


/tmp/ipykernel_103096/2274381349.py:41: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig("figG1_convergence.png", dpi=300)


Generating Figure 2: Efficiency vs Gust...


/tmp/ipykernel_103096/2274381349.py:47: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='g', y='efficiency', data=df, inner="quart", palette="muted")


Generating Figure 4: Density Analysis...
Generating Figure 6: Parameter Correlations...
Generating Figure 7: Stability...
Generating Figure 8: CDF...


/tmp/ipykernel_103096/2274381349.py:100: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig("figG8_cdf.png", dpi=300)



All analysis files generated successfully!
Files saved: figG1, figG2, figG4, figG6, figG7, figG8.


In [ ]:
import pandas as pd
import os

def split_massive_csv(input_file, rows_per_chunk=1000000):
    """
    Splits a large CSV into multiple smaller files.
    Default 1,000,000 rows usually results in ~40MB to 60MB files
    depending on the number of columns.
    """
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return

    # Create an output directory
    output_dir = "split_chunks"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Starting split of {input_file}...")

    # Use pandas chunksize to process file without loading it all into RAM
    reader = pd.read_csv(input_file, chunksize=rows_per_chunk)

    for i, chunk in enumerate(reader):
        chunk_filename = f"{output_dir}/aero_data_part_{i+1}.csv"

        # Save chunk
        chunk.to_csv(chunk_filename, index=False)

        # Get file size for confirmation
        file_size_mb = os.path.getsize(chunk_filename) / (1024 * 1024)
        print(f"Saved: {chunk_filename} | Size: {file_size_mb:.2f} MB | Rows: {len(chunk)}")

    print(f"\nSuccess! All chunks are stored in the '{output_dir}' folder.")

print("split_massive_csv function defined and ready.")

split_massive_csv function defined and ready.


In [ ]:
# Call the function to split the CSV file
# Using the default rows_per_chunk=1000000, which typically results in files 40-60MB,
# satisfying the user's request for files 100MB or lower.
split_massive_csv(TARGET_FILE)

Error: aero_sweep_results.csv not found.


Now that the data is split into manageable chunks, let's generate individual performance dashboards for each unique 'Gust Parameter ($G$)' value found in the dataset. This will help us analyze the performance trends within each specific sweep condition.

First, I'll create a helper function to load all the data into a single DataFrame. Then, I'll modify the `generate_dashboard` function to accept the loaded data and an optional `g_filter` parameter. This will allow us to reuse the plotting logic for both the overall dashboard and individual 'G' sweeps. Finally, I'll loop through each unique 'G' value to generate and save a separate dashboard for each.

In [ ]:
def load_aero_data():
    """
    Loads all aero data chunks from the 'split_chunks' directory into a single DataFrame.
    """
    print("Searching for data shards...")
    file_pattern = "split_chunks/aero_data_part_*.csv"
    files = glob.glob(file_pattern)

    if not files:
        print("No files found matching 'aero_data_part_*.csv'. Please check file names and directory.")
        return None

    print(f"Loading {len(files)} files into memory...")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f"Dataset Loaded: {len(df):,} total trials.")
    return df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Set visual style
sns.set_theme(style="darkgrid")
plt.rcParams['font.family'] = 'sans-serif'

# New helper function for directory management
def ensure_output_dir(base_dir="analysis_plots", sub_dir=""):
    """Ensures the output directory exists."""
    path = os.path.join(base_dir, sub_dir)
    os.makedirs(path, exist_ok=True)
    return path

def generate_dashboard(df_data=None, g_filter=None, base_output_dir="analysis_plots"):
    """
    Generates a performance dashboard. Can operate on a full dataset or a filtered subset.

    Args:
        df_data (pd.DataFrame, optional): The DataFrame to use. If None, data will be loaded.
        g_filter (float, optional): If provided, filters the DataFrame to this specific 'g' value.
        base_output_dir (str): Base directory to save dashboards.
    """
    if df_data is None:
        print("Searching for data shards...")
        file_pattern = "split_chunks/aero_data_part_*.csv"
        files = glob.glob(file_pattern)

        if not files:
            print("No files found matching 'aero_data_part_*.csv'. Please check file names and directory.")
            return

        print(f"Loading {len(files)} files into memory...")
        df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
        print(f"Dataset Loaded: {len(df):,} total trials.")
    else:
        df = df_data.copy()

    dashboard_output_dir = ensure_output_dir(base_output_dir, "dashboards")

    if g_filter is not None:
        df = df[df['g'] == g_filter].copy()
        if df.empty:
            print(f"No data found for G = {g_filter}. Skipping dashboard generation.")
            return
        dashboard_title = f'AV-32 Hyper-Sonic Performance Analysis (G={g_filter})\nTotal Trials: {len(df):,}'
        output_filename = f"Aero_Performance_Dashboard_G{g_filter}.png"
    else:
        dashboard_title = f'AV-32 Hyper-Sonic Performance Analysis\nTotal Trials: {len(df):,}'
        output_filename = "Aero_Performance_Dashboard.png"

    # Create figure with 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(dashboard_title, fontsize=20, fontweight='bold')

    # 2. GRAPH 1: EFFICIENCY DISTRIBUTION
    print("Generating distribution plot...")
    sns.histplot(df['efficiency'], kde=True, color='teal', bins=50, ax=axes[0, 0])
    axes[0, 0].set_title('Efficiency Distribution', fontsize=14)
    axes[0, 0].set_xlabel(r'Efficiency (eta)')
    axes[0, 0].set_ylabel('Frequency')

    # 3. GRAPH 2: GUST INTENSITY (G) VS EFFICIENCY (or Efficiency Distribution for single G) (Boxplot or Distribution)
    print("Generating second plot...")
    if g_filter is None:
        # Boxplot for overall dashboard
        sns.boxplot(x='g', y='efficiency', data=df, hue='g', palette='viridis', legend=False, ax=axes[0, 1])
        axes[0, 1].set_title('Impact of Gust Intensity ($G$) on Efficiency', fontsize=14)
        axes[0, 1].set_xlabel('Gust Parameter ($G$)')
        axes[0, 1].set_ylabel(r'Efficiency (eta)')
    else:
        # Distribution for single G value
        sns.histplot(df['efficiency'], kde=True, color='purple', bins=50, ax=axes[0, 1])
        axes[0, 1].set_title(f'Efficiency Distribution for G={g_filter}', fontsize=14)
        axes[0, 1].set_xlabel(r'Efficiency (eta)')
        axes[0, 1].set_ylabel('Frequency')

    # 4. GRAPH 3: PERFORMANCE STABILITY (Moving Average)
    print("Generating stability plot...")
    # We take a sample or decimate for the line plot if it's 10M to save memory/rendering time
    step = max(1, len(df) // 1000)
    sampled_df = df.iloc[::step].copy()
    sampled_df['rolling_avg'] = sampled_df['efficiency'].rolling(window=50).mean()

    axes[1, 0].plot(sampled_df['rolling_avg'], color='orange', linewidth=2)
    axes[1, 0].set_title('System Stability (50-Window Rolling Mean)', fontsize=14)
    axes[1, 0].set_xlabel('Sampled Trial Chronology')
    axes[1, 0].set_ylabel(r'eta (Avg)')

    # 5. GRAPH 4: CORRELATION MATRIX
    print("Generating correlation heatmap...")
    corr = df.corr(numeric_only=True)
    sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, ax=axes[1, 1])
    axes[1, 1].set_title('Parameter Correlation Matrix', fontsize=14)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Save the result
    output_path = os.path.join(dashboard_output_dir, output_filename)
    plt.savefig(output_path, dpi=300)
    print(f"Dashboard saved as {output_path}")
    plt.close(fig) # Close plot to free memory

if __name__ == "__main__":
    # This block is typically for script execution, but in Colab, cells run sequentially.
    # The main execution will be orchestrated in a separate cell below.
    pass

In [ ]:
print("Generating overall dashboard...")
full_df = load_aero_data()

if full_df is not None:
    # Generate the overall dashboard first
    generate_dashboard(df_data=full_df)

    # Now, generate dashboards for each unique 'g' value
    unique_g_values = sorted(full_df['g'].unique())
    print(f"\nGenerating dashboards for {len(unique_g_values)} unique G values: {unique_g_values}")

    for g_val in unique_g_values:
        print(f"\nGenerating dashboard for G = {g_val}...")
        generate_dashboard(df_data=full_df, g_filter=g_val)

    print("All dashboards generated successfully!")

Generating overall dashboard...
Searching for data shards...
No files found matching 'aero_data_part_*.csv'. Please check file names and directory.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob

# Style settings for academic-quality plots
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 16
})

def run_full_analysis():
    # 1. LOAD AND MERGE DATA
    print("Loading data shards...")
    # Corrected file pattern to look into 'split_chunks' directory
    files = sorted(glob.glob("split_chunks/aero_data_part_*.csv"))
    if not files:
        print("Error: No CSV files found. Please ensure the 'split_chunks' directory exists and contains the CSV parts.")
        return

    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f"Dataset ready: {len(df):,} trials across {df['g'].nunique()} gust levels.")

    # 2. CONVERGENCE PLOT (Mean Efficiency over Time)
    print("Generating Figure 1: Convergence...")
    plt.figure(figsize=(10, 6))
    df['rolling_mean'] = df['efficiency'].expanding().mean()
    plt.plot(df['rolling_mean'], color='#2c3e50', label='Global Expansion Mean')
    plt.axhline(y=df['efficiency'].mean(), color='r', linestyle='--', label='Final Mean')
    plt.title(r"Statistical Convergence of Efficiency (eta)") # Fixed escape sequence
    plt.xlabel("Trial Count")
    plt.ylabel(r"Cumulative Average eta") # Fixed escape sequence
    plt.legend()
    plt.savefig("figG1_convergence.png", dpi=300)
    plt.close()

    # 3. ETA VS GUST INTENSITY (Violin + Boxplot)
    print("Generating Figure 2: Efficiency vs Gust...")
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='g', y='efficiency', data=df, inner="quart", palette="muted")
    plt.title(r"Distribution of Efficiency (eta) by Gust Intensity ($G$)") # Fixed escape sequence
    plt.xlabel(r"Gust Intensity ($G$)") # Fixed escape sequence
    plt.ylabel(r"Efficiency (eta)") # Fixed escape sequence
    plt.savefig("figG2_eta_vs_G.png", dpi=300)
    plt.close()

    # 4. KERNEL DENSITY ESTIMATE (KDE)
    print("Generating Figure 4: Density Analysis...")
    plt.figure(figsize=(10, 6))
    for g_val in sorted(df['g'].unique()):
        subset = df[df['g'] == g_val]
        sns.kdeplot(subset['efficiency'], label=f'G={g_val}', fill=True, alpha=0.1)
    plt.title("Efficiency Density Profiles across Gust Conditions")
    plt.xlabel(r"Efficiency (eta)") # Fixed escape sequence
    plt.ylabel("Probability Density")
    plt.legend(title="Gust Levels")
    plt.savefig("figG4_kde.png", dpi=300)
    plt.close()

    # 5. CORRELATION HEATMAP
    print("Generating Figure 6: Parameter Correlations...")
    plt.figure(figsize=(8, 6))
    corr = df[['g', 'm', 'r', 'efficiency']].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Parameter Correlation Matrix")
    plt.savefig("figG6_heatmap.png", dpi=300)
    plt.close()

    # 6. STABILITY ANALYSIS (Variance window)
    print("Generating Figure 7: Stability...")
    plt.figure(figsize=(10, 6))
    stability = df['efficiency'].rolling(window=1000).std()
    plt.plot(stability, color='darkgreen', alpha=0.7)
    plt.title("System Variance Stability (1000-Trial Rolling Std Dev)")
    plt.xlabel("Trial Index")
    plt.ylabel(r"Standard Deviation ($\sigma$)") # Fixed escape sequence
    plt.savefig("figG7_stability.png", dpi=300)
    plt.close()

    # 7. CUMULATIVE DISTRIBUTION FUNCTION (CDF)
    print("Generating Figure 8: CDF...")
    plt.figure(figsize=(10, 6))
    for g_val in sorted(df['g'].unique()):
        subset = df[df['g'] == g_val]
        sorted_data = np.sort(subset['efficiency'])
        yvals = np.arange(len(sorted_data)) / float(len(sorted_data)-1)
        plt.plot(sorted_data, yvals, label=f'G={g_val}')
    plt.title("Cumulative Distribution of Efficiency")
    plt.xlabel(r"Efficiency (eta)") # Fixed escape sequence
    plt.ylabel(r"Probability $P(X \leq x)$") # Fixed escape sequence
    plt.legend()
    plt.savefig("figG8_cdf.png", dpi=300)
    plt.close()

    print("\nAll analysis files generated successfully!")
    print("Files saved: figG1, figG2, figG4, figG6, figG7, figG8.")

if __name__ == "__main__":
    run_full_analysis()

Loading data shards...
Error: No CSV files found. Please ensure the 'split_chunks' directory exists and contains the CSV parts.


In [ ]:
import shutil

def save_to_drive(file_path, drive_folder_name='AeroAnalysis_Plots'):
    """
    Saves a file to a specified folder in Google Drive.
    The Google Drive must already be mounted.
    """
    drive_path = os.path.join('/content/drive/MyDrive', drive_folder_name)
    os.makedirs(drive_path, exist_ok=True)

    destination_path = os.path.join(drive_path, os.path.basename(file_path))
    shutil.copy(file_path, destination_path)
    print(f"Saved '{os.path.basename(file_path)}' to Google Drive: {destination_path}")

Now, let's run the full analysis again and save the generated plots to a folder in your Google Drive.

In [ ]:
# Re-run the full analysis to generate plots
plot_files_generated = False

if 'run_full_analysis' in globals() and callable(globals()['run_full_analysis']):
    run_full_analysis()
    plot_files_generated = True
else:
    print("*******************************************************************************")
    print("WARNING: 'run_full_analysis' function is not defined in the current environment.")
    print("         Please ensure the cell 'uSCzaO6D4qkw' (which defines this function)")
    print("         has been executed successfully before running this cell.")
    print("         Skipping analysis re-run and plot saving due to missing function.")
    print("*******************************************************************************")

# List of generated plot files to save to Drive
plot_files = [
    "figG1_convergence.png",
    "figG2_eta_vs_G.png",
    "figG4_kde.png",
    "figG6_heatmap.png",
    "figG7_stability.png",
    "figG8_cdf.png"
]

# Save each plot to Google Drive, only if analysis was run
if plot_files_generated:
    for plot_file in plot_files:
        if os.path.exists(plot_file): # Check if the file actually exists
            save_to_drive(plot_file)
        else:
            print(f"Warning: Plot file '{plot_file}' not found. Skipping save to Drive.")
else:
    print("Skipping plot saving to Drive as analysis was not successfully run.")

Loading data shards...
Error: No CSV files found. Please ensure the 'split_chunks' directory exists and contains the CSV parts.
Saved 'figG1_convergence.png' to Google Drive: /content/drive/MyDrive/AeroAnalysis_Plots/figG1_convergence.png
Saved 'figG2_eta_vs_G.png' to Google Drive: /content/drive/MyDrive/AeroAnalysis_Plots/figG2_eta_vs_G.png
Saved 'figG4_kde.png' to Google Drive: /content/drive/MyDrive/AeroAnalysis_Plots/figG4_kde.png
Saved 'figG6_heatmap.png' to Google Drive: /content/drive/MyDrive/AeroAnalysis_Plots/figG6_heatmap.png
Saved 'figG7_stability.png' to Google Drive: /content/drive/MyDrive/AeroAnalysis_Plots/figG7_stability.png
Saved 'figG8_cdf.png' to Google Drive: /content/drive/MyDrive/AeroAnalysis_Plots/figG8_cdf.png


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
TARGET_FILE = 'aero_sweep_results.csv' # Assuming this is the original file name based on previous outputs

In [ ]:
# Re-run the split function to ensure data chunks are present
split_massive_csv(TARGET_FILE)

Error: aero_sweep_results.csv not found.


In [ ]:
# Now, run the full analysis again
run_full_analysis()

Loading data shards...
Error: No CSV files found. Please ensure the 'split_chunks' directory exists and contains the CSV parts.
